In [28]:
#Clean Text
def clean_text(comments):
    cleaned_comments=[]
    for comment in comments:
        for char in "?,.!'\"":
            comment = comment.replace(char,"")
        comment = comment.strip()
        comment = comment.lower()
        comment = comment.split()
        cleaned_comments.append(comment)
    return cleaned_comments

#comments= "Is this even real?"
#cleaned_comments = clean_text(comments)
#print(cleaned_comments)

In [29]:
#Building Vocab
def build_vocab(cleaned_comments):
    vocab=[]
    for comment in cleaned_comments:
        for word in comment:
            if word not in vocab:
                vocab.append(word)
    return vocab

#vocab = build_vocab(cleaned_comments)
#print(vocab)

In [30]:
def build_index(vocab):
    word_to_index={}
    count=0
    for word in vocab:
        word_to_index[word]= count
        count+=1
    return word_to_index

#word_to_index= build_index(vocab)
#print(word_to_index)

In [31]:
def tokenizer(comments):
    cleaned_comments=clean_text(comments)
    vocab=build_vocab(cleaned_comments)
    word_to_index= build_index(vocab)
    print(word_to_index)
    return cleaned_comments, word_to_index 

#tokenized_comments = tokenizer(comments) 

In [32]:
def vectorize(sentence,word_to_index):
    vector = [0]* len(word_to_index)
    for word in sentence:
        index = word_to_index[word]
        vector[index]= 1
    return vector

#vector = vectorize(word_to_index,vocab)
#print(vector)

In [33]:
#tuple with text and corresponding labels - reflection of training dataset
comments = [
    ("you are stupid and i hate you", "toxic"),
    ("this is a wonderful day", "not toxic"),
    ("i hate everything about this", "toxic"),
    ("you are absolutely wonderful", "not toxic"),
]

text=[text for text, label in comments ]
label=[label for text, label in comments]

cleaned_comments, word_to_index = tokenizer(text)
vector = [vectorize(sentence, word_to_index) for sentence in cleaned_comments ]
print(vector)

{'you': 0, 'are': 1, 'stupid': 2, 'and': 3, 'i': 4, 'hate': 5, 'this': 6, 'is': 7, 'a': 8, 'wonderful': 9, 'day': 10, 'everything': 11, 'about': 12, 'absolutely': 13}
[[1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0], [0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 0], [1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1]]


In [ ]:
def train_naive_bayes(cleaned_comments, label, word_to_index):
    total = len(label)
    classes = set(label)
    count =0
    class_probs={}
    for c in classes:
        for l in label:
            if l==c:
                count+=1
        class_probs[c]= count/total
        count=0
    
    toxic= []
    non_toxic=[]

    for comment, l in zip(cleaned_comments, label):
        if l=="toxic":
            toxic.append(comment)
        else:
            non_toxic.append(comment)

    toxic_word_counts = {}
    non_toxic_word_counts ={}
    for comment in toxic:
        for word in comment:
            if word in toxic_word_counts:
                toxic_word_counts[word]+=1
            else:
                toxic_word_counts[word] = 1
    
    for comment in non_toxic:
        for word in comment:
            if word in non_toxic_word_counts:
                non_toxic_word_counts[word]+=1
            else:
                non_toxic_word_counts[word] = 1

    #building toxic and non-toxic words prob class
    total_toxic_words = sum(toxic_word_counts.values())
    total_non_toxic_words = sum(non_toxic_word_counts.values())
    vocab_size = len(word_to_index)
    toxic_word_probs = {}
    non_toxic_word_probs ={}
    for word in word_to_index:
        toxic_word_probs[word]= (toxic_word_counts.get(word,0)+1)/(total_toxic_words+ vocab_size)
        non_toxic_word_probs[word]= (non_toxic_word_counts.get(word,0)+1)/(total_non_toxic_words+ vocab_size)

    return class_probs,toxic_word_probs, non_toxic_word_probs
        

class_probs, toxic_word_probs, non_toxic_word_probs =train_naive_bayes(cleaned_comments,label,word_to_index)
print("Toxic words class probabilities: ", toxic_word_probs)
print("Non-Toxic word class probabilities: ", non_toxic_word_probs)

{'not toxic': 0.5, 'toxic': 0.5}
Toxic words class probabilities:  {'you': 0.11538461538461539, 'are': 0.07692307692307693, 'stupid': 0.07692307692307693, 'and': 0.07692307692307693, 'i': 0.11538461538461539, 'hate': 0.11538461538461539, 'this': 0.07692307692307693, 'is': 0.038461538461538464, 'a': 0.038461538461538464, 'wonderful': 0.038461538461538464, 'day': 0.038461538461538464, 'everything': 0.07692307692307693, 'about': 0.07692307692307693, 'absolutely': 0.038461538461538464}
Non-Toxic word class probabilities:  {'you': 0.08695652173913043, 'are': 0.08695652173913043, 'stupid': 0.043478260869565216, 'and': 0.043478260869565216, 'i': 0.043478260869565216, 'hate': 0.043478260869565216, 'this': 0.08695652173913043, 'is': 0.08695652173913043, 'a': 0.08695652173913043, 'wonderful': 0.13043478260869565, 'day': 0.08695652173913043, 'everything': 0.043478260869565216, 'about': 0.043478260869565216, 'absolutely': 0.08695652173913043}


In [39]:
def predict(comment, class_probs, toxic_word_probs, non_toxic_word_probs, word_to_index):
    # 1. clean the comment
    cleaned_comment = clean_text([comment])
    cleaned_comment = cleaned_comment[0] 

    # 2. calculate log score for each class
    import math 
    toxic_class_score= math.log(class_probs["toxic"])
    non_toxic_class_score = math.log(class_probs["not toxic"])
    toxic_score= toxic_class_score
    non_toxic_score = non_toxic_class_score

    # 3. return whichever class has higher score
    for word in cleaned_comment:
        toxic_score += math.log(toxic_word_probs.get(word,1e-10))
        non_toxic_score +=math.log(non_toxic_word_probs.get(word,1e-10))

    if toxic_score>non_toxic_score:
        return "Toxic"
    else:
        return "Not Toxic"      

In [ ]:
print(predict("i hate you", class_probs, toxic_word_probs, non_toxic_word_probs, word_to_index))
print(predict("you are wonderful", class_probs, toxic_word_probs, non_toxic_word_probs, word_to_index))
print(predict("this is absolutely stupid", class_probs, toxic_word_probs, non_toxic_word_probs, word_to_index)) #the wordabsolutely is dominating word stupid so this mistakenly eneded up with Not Toxic class. This is training data bias.

Toxic
Not Toxic
Not Toxic
